In [6]:
TEAMS = [
    'cle', 'rav', 'cin', 'pit',
    'clt', 'htx', 'oti', 'jax',
    'den', 'kan', 'rai', 'sdg',
    'nwe', 'mia', 'nyj', 'buf',
    'min', 'gnb', 'chi', 'det',
    'tam', 'nor', 'atl', 'car',
    'dal', 'phi', 'was', 'nyg',
    'crd', 'ram', 'sfo', 'sea'
         ]

# Cleveland, Baltimore, Cincinatti, Pittsburgh
# Indianapolis, Houston, Tennessee, Jacksonville
# Denver, Kansas City, Las Vegas, Los Angeles Chargers
# New England, Miami, New York Jets, Buffalo

# Minnesota, Green Bay, Chicago, Detroit
# Tampa Bay, New Orleans, Atlanta, Carolina
# Dallas, Philadelphia, Washington, New York Giants
# Arizona, Los Angeles Rams, San Francisco, Seattle

In [7]:
"""
pro-football-reference - Page Fetch Test (pydoll)
--------------------------------------------------
Walks through the crawl path and prints the status code + first
500 chars of the final page HTML to confirm we can get through.

Usage:
    python pfr_roster_scraper.py

Requirements:
    pip install pydoll-python
    Chrome or Chromium must be installed on your machine.
"""

import asyncio
import nest_asyncio
import random
from pydoll.browser import Chrome
from pydoll.browser.options import ChromiumOptions
import pandas as pd
from bs4 import BeautifulSoup

# ── Config ────────────────────────────────────────────────────────────────────
MIN_DELAY = 10
MAX_DELAY = 18
# ──────────────────────────────────────────────────────────────────────────────

nest_asyncio.apply()

soup = None

async def main():
    options = ChromiumOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")

    async with Chrome(options=options) as browser:
        tab = await browser.start()

        url = "https://www.pro-football-reference.com/teams/nwe/2025_roster.htm"
        await tab.go_to(url)
        
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        print(f"    Waiting {delay:.1f}s before final request...\n")
        await asyncio.sleep(delay)

        resp = await tab.request.get(url)
        print(f"\n--- Result ---")
        print(f"Status code : {resp.status_code}")
        print(f"HTML preview:\n{resp.text[:100]}")

        # 1. Parse the HTML
        soup = BeautifulSoup(resp.text, 'html.parser')

        # 2. Locate the specific table by its ID
        roster_table = soup.find('table', id='roster')
        
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            comment_soup = BeautifulSoup(comment, 'html.parser')
            roster_table = comment_soup.find('table', id='roster')
            if roster_table:
                break

        # 3. Initialize your data list
        extracted_data = []

        if roster_table:
            # Look only within the 'tbody' of the targeted 'roster' table
            rows = roster_table.find('tbody').find_all('tr')
            
            for row in rows:
                player = row.find('td', {'data-stat': 'player'}).get_text(strip=True)
                player_id = row.find('td', {'data-stat': 'player'}).get('data-append-csv')
                pos = row.find('td', {'data-stat': 'pos'}).get_text(strip=True)
                av = row.find('td', {'data-stat': 'av'}).get_text(strip=True)
                
                year = 2025
                
                extracted_data.append({
                    'Year': year,
                    'PlayerID': player_id,
                    'Position': pos,
                    'Player': player,
                    'AV': av
                })

        # 4. Convert to DataFrame
        df = pd.DataFrame(extracted_data)
        print(df)



asyncio.run(main())

    Waiting 16.4s before final request...


--- Result ---
Status code : 200
HTML preview:

<!DOCTYPE html>
<html data-version="klecko-" data-root="/home/pfr/build" lang="en" class="no-js" >



/Users/arnav/.pyenv/versions/3.10.12/lib/python3.10/site-packages/bs4/__init__.py:868: RuntimeWarning: coroutine 'main' was never awaited
  self.object_was_parsed(o)


NameError: name 'Comment' is not defined

## Looping through the data

In [2]:
import asyncio
import nest_asyncio
import random
from pydoll.browser import Chrome
from pydoll.browser.options import ChromiumOptions
import pandas as pd
from bs4 import BeautifulSoup, Comment

In [9]:
TEAMS = [
    'cle', 'rav', 'cin', 'pit',
    'clt', 'htx', 'oti', 'jax',
    'den', 'kan', 'rai', 'sdg',
    'nwe', 'mia', 'nyj', 'buf',
    'min', 'gnb', 'chi', 'det',
    'tam', 'nor', 'atl', 'car',
    'dal', 'phi', 'was', 'nyg',
    'crd', 'ram', 'sfo', 'sea'
         ]
YEARS = [2020]
MIN_DELAY = 10
MAX_DELAY = 18

def parse_roster(html, year, team):
    """Parse roster table from HTML, handling commented-out tables."""
    soup = BeautifulSoup(html, 'html.parser')

    # Check comments first (Sports Reference hides tables in comments)
    roster_table = None
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        comment_soup = BeautifulSoup(comment, 'html.parser')
        roster_table = comment_soup.find('table', id='roster')
        if roster_table:
            break

    # Fallback to direct find
    if not roster_table:
        roster_table = soup.find('table', id='roster')

    if not roster_table:
        print(f"  WARNING: No roster table found for {team} {year}")
        return []

    extracted_data = []
    rows = roster_table.find('tbody').find_all('tr')
    for row in rows:
        # Skip spacer/header rows inside tbody
        if row.get('class') and 'thead' in row.get('class'):
            continue

        player_td = row.find('td', {'data-stat': 'player'})
        if not player_td:
            continue

        player    = player_td.get_text(strip=True)
        player_id = player_td.get('data-append-csv')
        pos       = row.find('td', {'data-stat': 'pos'}).get_text(strip=True)
        av        = row.find('td', {'data-stat': 'av'}).get_text(strip=True)

        extracted_data.append({
            'Year':     year,
            'Team':     team,
            'PlayerID': player_id,
            'Player':   player,
            'Position': pos,
            'AV':       av,
        })

    return extracted_data


async def scrape_roster(tab, team, year):
    """Navigate to and scrape a single roster page."""
    url = f"https://www.pro-football-reference.com/teams/{team}/{year}_roster.htm"
    print(f"  Scraping {team.upper()} {year}...")

    await tab.go_to(url)

    if team == 'cle':
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
    else:
        delay = random.uniform(3, 8)
    print(f"    Waiting {delay:.1f}s...")
    await asyncio.sleep(delay)

    resp = await tab.request.get(url)
    if resp.status_code != 200:
        print(f"  ERROR: Status {resp.status_code} for {team} {year}")
        return []

    return parse_roster(resp.text, year, team)

# IMPORTANT
nest_asyncio.apply()

async def main():
    options = ChromiumOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")

    all_data = []

    async with Chrome(options=options) as browser:
        tab = await browser.start()

        for year in YEARS:
            for team in TEAMS:
                rows = await scrape_roster(tab, team, year)
                all_data.extend(rows)
                print(f"    -> {len(rows)} players found")

    df = pd.DataFrame(all_data)
    print(df)
    file_name = f"{YEARS[0]}_av.csv"
    df.to_csv(file_name, index=False)
    return df

asyncio.run(main())

  Scraping CLE 2020...
    Waiting 11.5s...
    -> 71 players found
  Scraping RAV 2020...
    Waiting 5.6s...
    -> 75 players found
  Scraping CIN 2020...
    Waiting 5.4s...
    -> 69 players found
  Scraping PIT 2020...
    Waiting 4.8s...
    -> 67 players found
  Scraping CLT 2020...
    Waiting 4.9s...
    -> 69 players found
  Scraping HTX 2020...
    Waiting 7.0s...
    -> 70 players found
  Scraping OTI 2020...
    Waiting 3.2s...
    -> 74 players found
  Scraping JAX 2020...
    Waiting 4.6s...
    -> 81 players found
  Scraping DEN 2020...
    Waiting 6.5s...
    -> 75 players found
  Scraping KAN 2020...
    Waiting 3.7s...
    -> 70 players found
  Scraping RAI 2020...
    Waiting 4.5s...
    -> 70 players found
  Scraping SDG 2020...
    Waiting 5.6s...
    -> 71 players found
  Scraping NWE 2020...
    Waiting 6.5s...
    -> 70 players found
  Scraping MIA 2020...
    Waiting 5.3s...
    -> 63 players found
  Scraping NYJ 2020...
    Waiting 4.9s...
    -> 80 players 

,Year,Team,PlayerID,Player,Position,AV
0,2020,cle,BeckOd00,Odell Beckham Jr.,WR,4
1,2020,cle,BentEl00,Elijah Benton,S,0
2,2020,cle,BitoJo00,Joel Bitonio,LG,11
3,2020,cle,BradJa01,Ja'Marcus Bradley,WR,1
4,2020,cle,BrowEv00,Evan Brown,C,0
...,...,...,...,...,...,...
2266,2020,sea,WalkDA02,D'Andre Walker,LB,0
2267,2020,sea,WheeCh00,Chad Wheeler,C,0
2268,2020,sea,WillLu00,Luke Willson,TE,0
2269,2020,sea,WilsRu00,Russell Wilson,QB,17
